# RD04 — Regime robustness and falsification

Frozen RD03 models; no new regime-fitting hypotheses or model selection. Rebuild original **VD18 AM-T-HAR-Blend** density and score the previously report-only September diagnostic and the RD03 evaluation period. Strict RD03 reproduction gate. The old market history has already been repeatedly inspected: all results are retrospective / exploratory. No live forecast and no AM-T-HAR-Blend change.


In [1]:
import os,sys,json,tempfile,gc,time,subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir(): ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Run from this repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import regime_robustness as rd4
from data_engineering.modeling import regime_density_bridge as rd3
cfg=rd4.Specification()
SYNTHETIC=os.getenv('RD04_SYNTHETIC','0')=='1'
if SYNTHETIC:
    TEMP=tempfile.TemporaryDirectory(prefix='rd04-synthetic-')
    OUT=Path(TEMP.name)/'04'
else:
    rootval=os.getenv('DERIBIT_DATA_ROOT')
    if not rootval:raise RuntimeError('DERIBIT_DATA_ROOT not configured')
    STORE=Path(rootval)/'research'/'cs04a_feature_store'
    HIST=Path(rootval)/'history'/'price_5m'
    ARCHIVE=ROOT/'notebooks'/'Volatility_Dynamics'/'18'/'18_manifest.json'
    RD03_SCORES=ROOT/'notebooks'/'Regime_Discovery'/'03'/'03_scores.csv'
    OUT=ROOT/'notebooks'/'Regime_Discovery'/'04'
OUT.mkdir(parents=True,exist_ok=True)
print(rd4.VERSION,'synthetic:',SYNTHETIC,'outputs:',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
test=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_regime_robustness.py')],
                    cwd=str(ROOT),env=env,capture_output=True,text=True)
print(test.stdout[-2200:]);print(test.stderr[-1200:]);assert test.returncode==0,'RD04 invariant tests failed'


RD04-0.1.0 synthetic: False outputs: D:\Projects\deribit_market_data\notebooks\Regime_Discovery\04
..........                                                               [100%]
10 passed in 6.45s




In [2]:
all_rows=[];excluded={};split_audit={};archived_verified=False
if SYNTHETIC:
    rng=np.random.default_rng(2614)
    for asset in rd3.Specification().assets:
        for period,start,ndays in [('diagnostic','2026-09-01',16),('evaluation','2026-09-19',16)]:
            for h in cfg.horizons:
                for date in pd.date_range(start,periods=ndays,freq='D',tz='UTC'):
                    for slot in range(6):
                        variance=1e-6*(1+slot/5+.01*(date.day%10))
                        y=rng.normal(0,np.sqrt(variance))
                        for method in rd4.METHODS:
                            bias=0 if method=='AM_T_HAR_BLEND' else rng.normal(0,0.0001)
                            all_rows.append({'asset':asset,'state_time':date+pd.Timedelta(minutes=slot*30),
                                'period':period,'horizon_min':h,'method':method,
                                'nll':-4.5+bias,'up_brier':.25+bias/3,'variance':variance,
                                'realized_return':y,'up_probability':.5})
    archived_verified=False
    split_audit={'synthetic':True}
else:
    archive=json.loads(ARCHIVE.read_text(encoding='utf-8'))
    assert rd3.VERSION=='RD03-0.1.0','RD03 source version changed; freeze protocol first'
    params=rd3.confirm_parameters(archive,rd3.Specification())
    vdframe,split_audit=rd3.archived_split_frame(HIST,rd3.Specification())
    for asset in cfg.primary_assets+('SOL','XRP'):
        started=time.monotonic()
        path=STORE/f'{asset}_features_and_forward_magnitude.parquet'
        if not path.exists():excluded[asset]='CS04-A local feature store not found';continue
        required=list(dict.fromkeys(['state_time','feature_ready',*rd3.STATE_FEATURES,*rd3.RETURN_FEATURES]))
        data=pd.read_parquet(path,columns=required)
        frame=rd3.periods(rd3.merge_market_features(vdframe,data,asset),rd3.Specification())
        variances={h:rd3.vd18_particle_variances(frame,asset,h,params,
                    particles=rd3.Specification().particle_count,block=256) for h in cfg.horizons}
        fitted,info=rd3.prepare_methods(frame,variances,rd3.Specification())
        held=frame.rd_period.isin(cfg.periods).to_numpy()
        for h in cfg.horizons:
            v=variances[h][held]
            y=frame[f'R_{h}'].to_numpy(float)[held]
            assert np.isfinite(y).all() and (v>0).all()
            sigma=np.sqrt(v.mean(axis=1))
            for method,shift in fitted[h].items():
                outcome=rd3.normal_mixture_scores(y,v,shift[held]*sigma)
                all_rows.append(pd.DataFrame({'asset':asset,'state_time':frame.loc[held,'state_time'].to_numpy(),
                    'period':frame.loc[held,'rd_period'].to_numpy(),'horizon_min':h,'method':method,
                    'nll':outcome['nll'],'up_brier':outcome['up_brier'],
                    'variance':outcome['variance'],'realized_return':y,
                    'up_probability':outcome['up_probability']}))
        print(asset,'replayed',sum(held),'origins, seconds:',round(time.monotonic()-started,1))
        del frame,data,variances,fitted,info;gc.collect()
    if excluded:raise RuntimeError('Full matched four-asset audit required: '+str(excluded))
assert all_rows,'no origin scores'
origins=pd.concat([r if isinstance(r,pd.DataFrame) else pd.DataFrame([r]) for r in all_rows],ignore_index=True)
origins=rd4.validate_origin_scores(origins,cfg)
print('matched scored rows:',len(origins),'assets:',sorted(origins.asset.unique()))
if not SYNTHETIC:
    archive_reproduction=rd4.verify_rd03_archive(origins,pd.read_csv(RD03_SCORES))
    archived_verified=True
    print('RD03 archived evaluation reproduced; max NLL absolute error:',archive_reproduction.abs_nll_difference.max())
else:
    archive_reproduction=pd.DataFrame()


BTC replayed 1919 origins, seconds: 11.2
ETH replayed 1919 origins, seconds: 11.2
SOL replayed 1919 origins, seconds: 11.4
XRP replayed 1919 origins, seconds: 11.1
matched scored rows: 214928 assets: ['BTC', 'ETH', 'SOL', 'XRP']
RD03 archived evaluation reproduced; max NLL absolute error: 3.2862601528904634e-14


In [3]:
daily=rd4.paired_daily(origins)
time_results=rd4.time_stability(daily,cfg)
volatility_results,sigma_cuts=rd4.volatility_stress(origins,cfg)
family=rd4.primary_family(daily,cfg)
verdict=rd4.decision(time_results,family,cfg,archive_verified=archived_verified,synthetic=SYNTHETIC)
summary=(origins.groupby(['asset','period','horizon_min','method'],as_index=False)
  .agg(n_origins=('nll','size'),n_days=('date','nunique'),
       nll=('nll','mean'),up_brier=('up_brier','mean'),
       forecast_up=('up_probability','mean')))
display(summary.query('horizon_min==30').sort_values(['asset','period','nll']))
display(time_results.query('horizon_min==30 and metric=="nll" and slice=="full" and reference=="direct_mean"'))
display(family)
print(json.dumps(verdict,indent=2))


,asset,period,horizon_min,method,n_origins,n_days,nll,up_brier,forecast_up
16,BTC,diagnostic,30,latent_hmm_mean,864,18,-4.731027,0.249573,0.501596
17,BTC,diagnostic,30,latent_semi_markov_mean,864,18,-4.731008,0.249576,0.501581
19,BTC,diagnostic,30,observable_tree_mean,864,18,-4.730713,0.249606,0.501596
18,BTC,diagnostic,30,latent_static_mean,864,18,-4.730656,0.249628,0.501767
15,BTC,diagnostic,30,direct_mean,864,18,-4.730651,0.249631,0.501695
20,BTC,diagnostic,30,sign_only_mean,864,18,-4.729135,0.250004,0.502035
14,BTC,diagnostic,30,AM_T_HAR_BLEND,864,18,-4.728887,0.250000,0.500000
47,BTC,evaluation,30,observable_tree_mean,1055,22,-4.831488,0.249025,0.499484
46,BTC,evaluation,30,latent_static_mean,1055,22,-4.831041,0.249262,0.499430
43,BTC,evaluation,30,direct_mean,1055,22,-4.831034,0.249267,0.499356


,asset,horizon_min,method,reference,period,slice,metric,n_days,n_origins,delta_equal_day,ci025,ci975,favorable_days,status
348,BTC,30,latent_hmm_mean,direct_mean,diagnostic,full,nll,18,864,-0.000376,-0.000938,0.000225,11,EXPLORATORY
354,BTC,30,latent_hmm_mean,direct_mean,evaluation,full,nll,22,1055,0.000131,-0.000343,0.000579,11,EXPLORATORY
384,BTC,30,latent_semi_markov_mean,direct_mean,diagnostic,full,nll,18,864,-0.000357,-0.000978,0.000241,11,EXPLORATORY
390,BTC,30,latent_semi_markov_mean,direct_mean,evaluation,full,nll,22,1055,0.000118,-0.000343,0.000615,10,EXPLORATORY
420,BTC,30,latent_static_mean,direct_mean,diagnostic,full,nll,18,864,-0.000005,-0.000015,0.000004,9,EXPLORATORY
426,BTC,30,latent_static_mean,direct_mean,evaluation,full,nll,22,1055,-0.000007,-0.000016,0.000002,16,EXPLORATORY
444,BTC,30,observable_tree_mean,direct_mean,diagnostic,full,nll,18,864,-0.000063,-0.000502,0.000352,9,EXPLORATORY
450,BTC,30,observable_tree_mean,direct_mean,evaluation,full,nll,22,1055,-0.000456,-0.000947,0.000037,12,EXPLORATORY
972,ETH,30,latent_hmm_mean,direct_mean,diagnostic,full,nll,18,864,-0.000001,-0.000652,0.000691,9,EXPLORATORY
978,ETH,30,latent_hmm_mean,direct_mean,evaluation,full,nll,22,1055,0.000877,0.000505,0.001379,2,EXPLORATORY


,asset,method,reference,period,horizon_min,n_days,delta_equal_day,bonferroni_upper_one_sided,familywise_alpha_per_comparison,familywise_screen_pass
0,BTC,latent_hmm_mean,direct_mean,diagnostic,30,18,-0.000376,0.000318,0.008333,False
1,BTC,latent_semi_markov_mean,direct_mean,diagnostic,30,18,-0.000357,0.000386,0.008333,False
2,BTC,observable_tree_mean,direct_mean,diagnostic,30,18,-0.000063,0.000448,0.008333,False
3,ETH,latent_hmm_mean,direct_mean,diagnostic,30,18,-0.000001,0.000833,0.008333,False
4,ETH,latent_semi_markov_mean,direct_mean,diagnostic,30,18,-0.000008,0.000804,0.008333,False
5,ETH,observable_tree_mean,direct_mean,diagnostic,30,18,0.000045,0.001501,0.008333,False


{
  "status": "REPEATEDLY_INSPECTED_HISTORICAL_ROBUSTNESS",
  "verdict": "NO_STABLE_INCREMENTAL_REGIME_EVIDENCE",
  "methods": {
    "observable_tree_mean": {
      "all_primary_windows_and_baselines_nominally_favorable": false,
      "n_nominal_favorable": 0,
      "n_required": 8
    },
    "latent_hmm_mean": {
      "all_primary_windows_and_baselines_nominally_favorable": false,
      "n_nominal_favorable": 0,
      "n_required": 8
    },
    "latent_semi_markov_mean": {
      "all_primary_windows_and_baselines_nominally_favorable": false,
      "n_nominal_favorable": 0,
      "n_required": 8
    }
  },
  "primary_family_tests": 6,
  "multiplicity_adjusted_confirmation": false,
  "prospective_confirmation": false,
  "AM_T_HAR_Blend_modified": false,
  "note": "Daily bootstrap and nominal intervals are descriptive after many previous searches. RD04 does not confer clean confirmation or authorize trading."
}


In [4]:
# Safe-to-share aggregates only. Individual origin losses, realized returns and
# variance particles stay in RAM and MUST NOT be added to the git repository.
files={
  '04_daily_pairs.csv':daily,
  '04_temporal_stability.csv':time_results,
  '04_volatility_stress.csv':volatility_results,
  '04_sigma_terciles.csv':sigma_cuts,
  '04_primary_multiplicity.csv':family,
  '04_rd03_reproduction.csv':archive_reproduction,
  '04_summary.csv':summary,
}
for name,value in files.items():value.to_csv(OUT/name,index=False)
manifest={'module':'RD04','version':rd4.VERSION,'spec':rd4.registration(),
    'source_RD03':'RD03-0.1.0','source_VD18':'VD18-0.1.0',
    'synthetic':SYNTHETIC,'assets_evaluated':sorted(origins.asset.unique()),
    'excluded':excluded,'vd18_archive_split':split_audit,
    'rd03_exact_archived_score_identity':archived_verified,
    'historical_data_repeatedly_inspected':True,
    'no_independent_confirmation':True,'AM_T_HAR_Blend_modified':False,
    'no_model_reselection':True}
(OUT/'04_manifest.json').write_text(json.dumps(manifest,indent=2,default=str),encoding='utf-8')
(OUT/'04_decision.json').write_text(json.dumps(verdict,indent=2),encoding='utf-8')
assert all((OUT/f).exists() for f in list(files)+['04_manifest.json','04_decision.json'])
print('Saved nine aggregate outputs:',OUT)


Saved nine aggregate outputs: D:\Projects\deribit_market_data\notebooks\Regime_Discovery\04


## Interpretation limits

- **RD03 already failed its historical primary incremental regime checks.** RD04 asks whether this negative/weak result persists across chronological windows and ex-ante volatility bands, not whether another model can be tuned to win.
- September diagnostic is held out *from these frozen RD03 response fits*, but the broader project has already examined this market history. Neither period is a prospective confirmation sample.
- Bonferroni bootstrap upper bounds screen six comparisons only; prior model-selection bias is **not** repaired. Day-bootstrap CIs may underrepresent multi-day clustering; stress buckets can be sparse.
- A semi-Markov duration overlay is not a jointly fitted HSMM. RD04 freezes the same VD18 variance particles for each competitor, tests only return-location overlays, and does not authorize trading or alter baseline models.
